# Exercise20261015

學號：413180233

本作業包含隨機小數的二進位表示、Python 任意精度整數與中文大寫數字轉換，以及使用 iterator 製作九九乘法表。

## 習題一：輸出隨機小數的前 20 位二進位表示

使用 `random.random()` 產生介於 0（含）與 1（不含）的小數 `r`。每次將小數部分乘以 2，取整數部分作為下一個二進位位元，再減去該整數部分繼續計算。依題目要求使用 `while` 迴圈取得 20 位。

In [ ]:
import random

r = random.random()
fraction = r
bits = ''
count = 0

while count < 20:
    fraction *= 2
    bit = int(fraction)
    bits += str(bit)
    fraction -= bit
    count += 1

print(f'假如 r = {r}')
print(f'r ≈ 0.{bits}₂')

### 原理說明

二進位小數的每一位可用「乘以 2，取整數部分」求出。若乘完後結果大於或等於 1，該位就是 1，並將 1 扣除後繼續；若小於 1，該位就是 0。因為只取 20 位，所以結果是近似值，不一定能完整表示原本的浮點數。每次執行產生的 `r` 通常不同，因此輸出也會不同。

## 習題二之一：觀察 Python 任意精度整數

`int` 是任意精度整數，數值不受固定 32 位或 64 位的整數範圍限制；只要記憶體足夠，就能儲存非常大的整數。`bit_length()` 回傳表示該整數絕對值所需的二進位位數（不包含符號位，也不計前導零）。`sys.getsizeof()` 回傳物件本身所占的大小，單位是 bytes，包含物件標頭等實作成本。`PyLongObject` 是 CPython 用來實作 Python 整數的內部結構名稱；它會以多個數字位元組／digit 儲存大整數，因此記憶體用量會隨數值大小增加。實際大小與 CPython 版本、平台有關。

In [ ]:
import sys

n = 10**68
print(n)
print(f'bit_length = {n.bit_length()}')
print(f'memory = {sys.getsizeof(n)} bytes')

### 三者之間的關係

- `bit_length()`：描述整數數值需要多少個二進位位元來表示。
- `sys.getsizeof()`：描述整數物件在目前 Python 實作中占用的記憶體大小，不等於數值的 bit 數直接除以 8，因為還包含物件管理成本。
- `PyLongObject`：CPython 內部實際表示整數的資料結構，`bit_length()` 與物件記憶體大小都和它的儲存方式有關。

因此，數值越大通常需要更多內部 digit，記憶體大小也會增加；但具體 bytes 數不是所有 Python 平台都固定相同。

## 習題二之二：將正整數轉換成中文大寫數字

以下程式支援 `1 ≤ n ≤ 10**68`。先將數字每四位分成一組，再依照萬、億、兆等單位由高到低轉換。數字使用財務大寫：壹、貳、參、肆、伍、陸、柒、捌、玖；位名使用拾、佰、仟。

In [ ]:
DIGITS = {0: '零', 1: '壹', 2: '貳', 3: '參', 4: '肆',
          5: '伍', 6: '陸', 7: '柒', 8: '捌', 9: '玖'}
SMALL_UNITS = ['', '拾', '佰', '仟']
LARGE_UNITS = ['', '萬', '億', '兆', '京', '垓', '秭', '穰',
               '溝', '澗', '正', '載', '極', '恆河沙', '阿僧祇',
               '那由他', '不可思議', '無量大數']

def four_digit_to_chinese(number):
    """將 0 到 9999 的整數轉成中文大寫，不在組內插入多餘的零。"""
    if number == 0:
        return ''
    result = ''
    zero_pending = False
    digits = [number // 1000, (number // 100) % 10,
              (number // 10) % 10, number % 10]
    for index, digit in enumerate(digits):
        position = 3 - index
        if digit == 0:
            if result and any(d != 0 for d in digits[index + 1:]):
                zero_pending = True
        else:
            if zero_pending:
                result += '零'
                zero_pending = False
            result += DIGITS[digit] + SMALL_UNITS[position]
    return result

def integer_to_chinese_upper(number):
    if not isinstance(number, int) or isinstance(number, bool):
        raise TypeError('輸入必須是整數')
    if number < 1 or number > 10**68:
        raise ValueError('請輸入 1 到 10**68 之間的正整數')

    groups = []
    while number > 0:
        groups.append(number % 10000)
        number //= 10000

    parts = []
    index = len(groups) - 1
    zero_between = False
    while index >= 0:
        group = groups[index]
        if group == 0:
            if parts:
                zero_between = True
        else:
            # 若中間有整組零，或目前組不足四位且前面已有內容，補一個零。
            if parts and (zero_between or group < 1000):
                if not parts[-1].endswith('零'):
                    parts.append('零')
            parts.append(four_digit_to_chinese(group) + LARGE_UNITS[index])
            zero_between = False
        index -= 1

    result = ''.join(parts).replace('零零', '零').rstrip('零')
    return result

n = int(input('請輸入正整數（1 到 10**68）：'))
print(integer_to_chinese_upper(n))

### 範例測試

題目範例 `31415926` 應輸出：`參仟壹佰肆拾壹萬伍仟玖佰貳拾陸`。

In [ ]:
print(integer_to_chinese_upper(31415926))
print(integer_to_chinese_upper(10000))
print(integer_to_chinese_upper(10001))
print(integer_to_chinese_upper(10**68))

## 習題三：使用 iterable、iter()、next()、while 與 StopIteration 製作九九乘法表

此題不使用 `for` 或 `range`。先用 `iter()` 建立迭代器，再透過 `next()` 逐一取出 1 到 9；當迭代器沒有資料時，`next()` 會引發 `StopIteration`，由 `except` 結束迴圈。

In [ ]:
rows = iter([1, 2, 3, 4, 5, 6, 7, 8, 9])
while True:
    try:
        i = next(rows)
    except StopIteration:
        break

    columns = iter([1, 2, 3, 4, 5, 6, 7, 8, 9])
    while True:
        try:
            j = next(columns)
        except StopIteration:
            break
        print(f'{i}×{j}={i*j:2}', end='  ')
    print()